# Embedding
  
Das Notebook dient dazu per:
  
- **CLIP**          CLIP -> 
- **AnyLoc**        DINOv2 -> Feature Aggregation -> Descriptor -> Retrival (Github: https://github.com/AnyLoc/Revisit-Anything.git)
- **EigenPlaces**   Backbone -> VPR-Descriptor -> Retrival (Github: https://github.com/gmberton/EigenPlaces.git)
- **MixVPR**        noch keine Idee (Mixed ansatz)
  
die Bilder in Vectorinformationen zu embedden


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
import sys


plt.rcParams["figure.dpi"] = 300

def find_project_root():
    for dir in (Path.cwd(), *Path.cwd().parents):
        if (dir / "config.yaml").exists():
            return dir
    raise FileNotFoundError("Projektroot nicht gefunden")


def find_upwards(name):
    # Erlaubt, das Notebook aus eval/ oder aus dem Projektwurzelverzeichnis
    # zu starten, ohne Pfade anzupassen.
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / name).exists():
            return d / name
    return None


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.run_guard import embedding_fingerprint, print_run_header, short_hash, write_fingerprint, validate_config


CFG_FILE = find_upwards("config.yaml")
assert CFG_FILE, "config.yaml nicht gefunden (liegt im Projektwurzelverzeichnis)."
CFG = yaml.safe_load(CFG_FILE.read_text())
validate_config(CFG)

METHOD = CFG["vpr"]["method"]
MODEL_ID = CFG["vpr"]["models"][METHOD]


N_IMAGES = None
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGE_PATH = Path(CFG["img_download_path"]).expanduser()
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
METHOD_DIR = EMBEDDING_DIR / f"{METHOD}"
METHOD_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH_META = PROCESSED_DIR / "metadata.parquet"

# Erzeugt nur die Baseline. Die adaptierte Variante schreibt 05, und
# vpr.adapter waehlt in 06/07 aus, welche der beiden ausgewertet wird.
EMBEDDING_NAME = METHOD

metadata = pd.read_parquet(DATA_PATH_META)
embedding_metadata = metadata[metadata["split"].isin(["train", "database", "query"])].copy()
embedding_metadata = embedding_metadata.reset_index(drop=True)
embedding_path = METHOD_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = METHOD_DIR / f"{EMBEDDING_NAME}_metadata.parquet"


# A GPU makes the image encoder roughly 20x faster, but nothing here *needs* one.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
    
BATCH_SIZE = CFG["vpr"]["embed_batch_size"]


# Wie viele Bilder eingebettet werden sollen
if N_IMAGES is not None:

    if N_IMAGES <= 0:
        raise ValueError("N_IMAGES muss None oder größer Null sein")
    
    if N_IMAGES > len(embedding_metadata):
        raise ValueError(
            f"N_IMAGES darf nicht größer als {len(embedding_metadata)} sein ist aber {N_IMAGES} "
        )
    embedding_metadata = embedding_metadata.iloc[:N_IMAGES].copy()


image_paths = [IMAGE_PATH / f"{i}.jpg" for i in embedding_metadata["image_id"]]

exists = np.array([p.exists() for p in image_paths])
if not exists.all():
    print(
        f"WARNUNG: {(~exists).sum():,} von {len(exists):,} Bildern fehlen "
        f"-- werden uebersprungen."
    )
    embedding_metadata = embedding_metadata[exists].reset_index(drop=True)
    image_paths = [p for p, ok in zip(image_paths, exists) if ok]


print(f"Bilder:             {len(metadata):,}")
print(f"Bildordner:         {IMAGE_PATH}")
print(f"Embedding-Ordner:   {EMBEDDING_DIR}")
print(f"running on:         {DEVICE}")
print(f"batch size:         {BATCH_SIZE}")
print(f"Method:             {METHOD}")
print(f"MODEL_ID:           {MODEL_ID}")


# Modell laden
  
**MODEL_REVISION** = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268" for _reloading_ Modell



In [ ]:
MODEL_REVISION = CFG["vpr"].get("clip", {}).get("revision")


def from_root(path):
    """
    Relative Pfade aus der config.yaml gegen den Projektroot aufloesen.

    resolve() allein haengt am Arbeitsverzeichnis -- in Jupyter ist das
    notebooks/, nicht der Projektroot.
    """
    path = Path(path).expanduser()
    return path if path.is_absolute() else (PROJECT_ROOT / path)


if METHOD == "clip":
    from src.models.clip import CLIPEmbedder
    embedder = CLIPEmbedder(model_id = MODEL_ID, device = DEVICE, revision = MODEL_REVISION, num_workers=8, use_amp=True)

elif METHOD == "anyloc":
    from src.models.anyloc import AnyLocEmbedder

    acfg = CFG["vpr"]["anyloc"]
    embedder = AnyLocEmbedder(
        model_id=MODEL_ID,
        device=DEVICE,
        repo_path=from_root(acfg["repo_path"]),
        vocabulary_domain=acfg["vocabulary_domain"],
        desc_layer=acfg["desc_layer"],
        desc_facet=acfg["desc_facet"],
        num_clusters=acfg["num_clusters"],
        image_size=acfg["image_size"],
        pca_dim=acfg.get("pca_dim"),
    )
    if acfg.get("pca_dim"):
        db_paths = [
            IMAGE_PATH / f"{i}.jpg"
            for i in embedding_metadata.loc[
                embedding_metadata["split"] == "train", "image_id"
            ]
        ]
        embedder.fit_pca(db_paths, n_images=acfg["pca_fit_images"])

elif METHOD == "eigenplaces":
    from src.models.eigenplaces import EigenPlacesEmbedder

    ecfg = CFG["vpr"]["eigenplaces"]
    embedder = EigenPlacesEmbedder(
        model_id=MODEL_ID,
        device=DEVICE,
        fc_output_dim=ecfg["fc_output_dim"],
        image_size=ecfg["image_size"],
    )


elif METHOD == "mixvpr":
    from src.models.mixvpr import MixEmbedder

    mcfg = CFG["vpr"]["mixvpr"]
    embedder = MixEmbedder(
        model_id=MODEL_ID,
        device=DEVICE,
        repo_path=from_root(mcfg["repo_path"]),
        weights=from_root(mcfg["weights"]),
        image_size=mcfg["image_size"],
        agg_config=mcfg["agg_config"],
    )

elif METHOD == "megaloc":
    from src.models.megaloc import MegaLocEmbedder

    embedder = MegaLocEmbedder(device=DEVICE)

else:
    raise ValueError(f"Unbekannte METHOD = {METHOD}")


EMBEDDING_DIM = embedder.embedding_dim

FINGERPRINT = embedding_fingerprint(CFG, METHOD, "none", embedding_metadata)
RUN_HASH = short_hash(FINGERPRINT)

print_run_header(CFG, "04_embeddings", device=DEVICE, run_hash=RUN_HASH)
print(f"Embedding dimension: {EMBEDDING_DIM}")


# Embedding Creation

In [ ]:
embeddings = embedder.embed_images(
    image_paths,
    batch_size=BATCH_SIZE,
    # Run-Hash im Namen, damit kein Teil-Checkpoint aus einem Lauf mit
    # anderen Einstellungen fortgesetzt wird.
    checkpoint_path=METHOD_DIR / f"{EMBEDDING_NAME}_{RUN_HASH}.partial.npy",
    checkpoint_every=500,
)

assert len(embeddings) == len(embedding_metadata)
assert embeddings.shape[1] == EMBEDDING_DIM
assert np.isfinite(embeddings).all()
norms = np.linalg.norm(embeddings, axis=1)

print(f"Shape:                 {embeddings.shape}")
print(f"Dtype:                 {embeddings.dtype}")
print(f"Normalized Minimum:    {norms.min():.2f}")
print(f"Normalized Maximum:    {norms.max():.2f}")
print(f"Normalized Mittelwert: {norms.mean():.2f}")


# Embedding Speichern


In [ ]:
np.save( embedding_path, embeddings)
embedding_metadata.to_parquet(metadata_path, index = False)

write_fingerprint(
    embedding_path,
    FINGERPRINT,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    n_images=len(embedding_metadata),
)

# Teil-Checkpoint aufraeumen, er ist so gross wie das Ergebnis selbst.
(METHOD_DIR / f"{EMBEDDING_NAME}_{RUN_HASH}.partial.npy").unlink(missing_ok=True)

print(f"Embeddings gespeichert in:  {embedding_path}")
print(f"Metadaten gespeichert in:   {metadata_path}")
print(f"Fingerabdruck:              {RUN_HASH}")